In [2]:
#!/usr/bin/env python3
"""
phrase_to_abstract.py

Given a phrase and its extracted important terms, returns a high-level
abstract concept by disambiguating each term via Lesk, then finding
their lowest common hypernym in WordNet.

Requirements:
    pip install nltk

First run will download WordNet automatically.
"""

from typing import Iterable, Optional, Set
import warnings

import nltk
from nltk.corpus import wordnet as wn
from nltk.wsd import lesk
from nltk.corpus.reader.wordnet import Synset as WordNetSynset

def _ensure_wordnet():
    try:
        wn.ensure_loaded()
    except LookupError:
        nltk.download('wordnet')
        nltk.download('omw-1.4')
        wn.ensure_loaded()

def _hypernym_closure(syn):
    seen = set()
    stack = [syn]
    while stack:
        cur = stack.pop()
        if cur in seen:
            continue
        seen.add(cur)
        stack.extend(cur.hypernyms())
        stack.extend(cur.instance_hypernyms())
    return seen

def _deepest_synset(syns: Set[WordNetSynset]) -> Optional[WordNetSynset]:
    best, best_depth = None, -1
    for s in syns:
        try:
            d = s.min_depth()
        except Exception:
            d = -1
        if d > best_depth:
            best, best_depth = s, d
    return best

def phrase_to_abstract(
    phrase: str,
    terms: Iterable[str],
    return_pretty: bool = True
) -> Optional[str]:
    """
    Returns the lowest common hypernym (most specific), disambiguated
    in context of `phrase` over `terms`.

    phrase : the full text context
    terms  : list of important words from that phrase
    return_pretty : if True, returns synset.name(), else Synset obj
    """
    _ensure_wordnet()

    selected_synsets = []
    for w in terms:
        syn = lesk(phrase.split(), w)
        if syn:
            selected_synsets.append(syn)
        else:
            all_syn = wn.synsets(w)
            if all_syn:
                selected_synsets.append(all_syn[0])
    if not selected_synsets:
        if return_pretty:
            warnings.warn("No synsets found for any term")
        return None

    closures = [ _hypernym_closure(s) | {s} for s in selected_synsets ]
    commons = set.intersection(*closures)

    if not commons:
        # 1. Try pairwise LCH across selected_synsets
        pair_lchs = []
        for i, s1 in enumerate(selected_synsets):
            for s2 in selected_synsets[i+1:]:
                common = set(_hypernym_closure(s1) | {s1}) & set(_hypernym_closure(s2) | {s2})
                if common:
                    pair_lchs.append(_deepest_synset(common))
        pair_lchs = [s for s in pair_lchs if s]
        if pair_lchs:
            # pick the deepest among pairwise
            lch = sorted(pair_lchs, key=lambda s: s.min_depth(), reverse=True)[0]

    lch = _deepest_synset(commons)
    if not lch:
        return None

    return (lch.name() if return_pretty else lch)

if __name__ == "__main__":
    phrase = "what cinema goers made of this in the 30s, I can only imagine."
    terms = ["imagine", "cinema", "goers"]
    concept = phrase_to_abstract(phrase, terms)
    print(f"Phrase: {phrase!r}")
    print(f"Terms: {terms}")
    print(f"Abstract concept: {concept}")


Phrase: 'what cinema goers made of this in the 30s, I can only imagine.'
Terms: ['imagine', 'cinema', 'goers']
Abstract concept: None
